# Word BoW and TF-IDF with a pairwise classifier

Each text of a pair becomes a word BoW or TF-IDF vector; the two vectors of a pair are combined
into one feature vector (their absolute difference or their element-wise product) and one of six
classifiers decides whether the texts were written by the same author. The vectorizer and the
classifier hyperparameters are tuned together with Optuna on the validation pairs, the final model
is trained on the training and validation pairs and evaluated once on the test pairs.

One run tunes ONE classifier for one representation and one combination. A restarted run continues
where it stopped: the study lives in a SQLite file and a finished run is not repeated. A study is
never continued with other settings than the ones it was started with.

In [ ]:
SMOKE = False  # True - the first rows of every data set, files of their own
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the sampler; the classifiers keep classifiers.SEED
RUN_TAG = ""  # label in the file names of a repeated run
REPRESENTATION = "tfidf"  # "tfidf" or "bow"
COMBINATION = "absolute_difference"  # "absolute_difference" or "element-wise"
CLASSIFIER = "logistic_regression"  # one of classifiers.CLASSIFIER_NAMES
N_JOBS = 1  # threads of kNN, random forest and XGBoost - no effect on the results

Standard libraries

In [ ]:
import logging
import re
import tempfile
import time
from functools import partial
from itertools import islice
from pathlib import Path

Models, data, plots and Optuna

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import seaborn as sns

# numpy 1.26 has no public function that returns this list
from numpy.core._multiarray_umath import (
    __cpu_dispatch__ as numpy_simd_extensions,
)
from numpy.core._multiarray_umath import (
    __cpu_features__ as cpu_has_feature,
)
from optuna.trial import TrialState
from sklearn.base import BaseEstimator
from sklearn.metrics import confusion_matrix, f1_score

The shared modules of the thesis

In [ ]:
from avcore import (
    classifiers,
    data,
    evaluation,
    features,
    progress,
    runs,
    tracking,
    tuning,
)

# Configuration

The vectorizer search space of each representation, every entry suggested in this order

In [ ]:
VECTORIZER_SEARCH_SPACES = {
    "tfidf": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "lowercase": ("categorical", False, True),
        "sublinear_tf": ("categorical", True, False),
    },
    "bow": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "max_df": ("float", 0.7, 1.0),
        "lowercase": ("categorical", False, True),
        "binary": ("categorical", False, True),
    },
}

The function that builds the features of each representation, and its part of the run name

In [ ]:
FEATURE_BUILDERS = {
    "tfidf": features.build_tfidf_features,
    "bow": features.build_bow_features,
}
RUN_NAME_PARTS = {"tfidf": "word_TF-IDF", "bow": "word_BOW"}

The two combinations of the vectors of a pair, and their names in the result tables

In [ ]:
COMBINE_FUNCTIONS = {
    "absolute_difference": features.compute_absolute_difference_features,
    "element-wise": features.compute_element_wise_features,
}
COMBINATION_TABLE_NAMES = {
    "absolute_difference": "absolute_difference",
    "element-wise": "elementwise_product",
}

A parameter outside these lists stops the run here

In [ ]:
if REPRESENTATION not in VECTORIZER_SEARCH_SPACES:
    raise ValueError(f"Unknown REPRESENTATION {REPRESENTATION!r}.")
if COMBINATION not in COMBINE_FUNCTIONS:
    raise ValueError(f"Unknown COMBINATION {COMBINATION!r}.")
if CLASSIFIER not in classifiers.CLASSIFIER_NAMES:
    raise ValueError(f"Unknown CLASSIFIER {CLASSIFIER!r}.")

The settings of this run

In [ ]:
VECTORIZER_SEARCH_SPACE = VECTORIZER_SEARCH_SPACES[REPRESENTATION]
BUILD_FEATURES = FEATURE_BUILDERS[REPRESENTATION]
COMBINE_PAIR_FEATURES = COMBINE_FUNCTIONS[COMBINATION]
CLASSIFIER_SEARCH_SPACE = classifiers.CLASSIFIER_SEARCH_SPACES[CLASSIFIER]
# every lowercase choice of the search space (its first item is the kind), each gets token counts
LOWERCASE_VALUES = VECTORIZER_SEARCH_SPACE["lowercase"][1:]
TRAINING_SECONDS_ATTR = "training_seconds"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "traditional",
    "model": "none",
    "representation": REPRESENTATION,
    "combination": COMBINATION_TABLE_NAMES[COMBINATION],
    "classifier": CLASSIFIER,
}

The smoke run - kNN needs more training pairs than its largest number of neighbours (50)

In [ ]:
SMOKE_ROWS = {"train": 120, "validation": 40, "test": 40}

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
CLEANED_PATHS = {
    "train": DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE,
    "validation": DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE,
    "test": DATA_DIR / "processed" / data.TEST_CLEANED_FILE,
}

The run name - it is part of every file name of the run

In [ ]:
run_name_parts = [DEVELOPER_INITIALS, "traditional"]
run_name_parts += [RUN_NAME_PARTS[REPRESENTATION], COMBINATION, CLASSIFIER]
if SMOKE:
    run_name_parts.append("smoke")
if RUN_TAG:
    run_name_parts.append(str(RUN_TAG))
run_name_parts.append("v2")
RUN_NAME = "_".join(run_name_parts)

A run name that is not a plain file name stops the run

In [ ]:
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(
        f"Run name {RUN_NAME!r} is not a plain file name - change RUN_TAG.",
    )

Every file the run writes

In [ ]:
STUDY_PATH = DATA_DIR / "studies" / f"optuna_study_{RUN_NAME}.db"
VECTORIZER_PATH = DATA_DIR / "models" / f"{RUN_NAME}_best_vectorizer.pkl"
MODEL_PATH = DATA_DIR / "models" / f"{RUN_NAME}_best_model.pkl"
# the result tables are built from results_v2, so a smoke run writes into a folder of its own
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIR = RESULTS_DIR / RUN_NAME
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Run context - the commit, the hashes of the lock file and of the data and the versions.
It refuses a repository with uncommitted or unpushed changes.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
run_context = tracking.collect_run_context(PROJECT_DIR, list(CLEANED_PATHS.values()))
print(f"Run {RUN_NAME}, commit {run_context.environment['commit'][:7]}")
# where the run stands, for watch_progress.py on dt - the run never reads it back
progress_file = progress.ProgressFile(
    Path.home() / "logs" / f"{RUN_NAME}.progress.json",
    RUN_NAME,
)
progress_file.start_stage("token counts")

Numpy uses the fastest instruction sets this CPU has (SIMD extensions such as AVX2 or AVX-512). Different instruction sets give slightly different np.log results and can change which terms max_features keeps. The list is therefore saved with the study, and a study is continued only on a CPU with the same list.

In [ ]:
simd_extensions_on_this_cpu = []
for extension in numpy_simd_extensions:
    if cpu_has_feature[extension]:
        simd_extensions_on_this_cpu.append(extension)
CPU_SIMD_FOUND = " ".join(simd_extensions_on_this_cpu)

# Settings

Everything that is fixed before the first trial. The pair counts are left out, because the data
file hashes below already fix the pairs - so a run with different settings stops here, before the
texts are counted.

In [ ]:
run_settings = {
    **RUN_IDENTITY,
    "smoke": SMOKE,
    "seed": SEED,
    "n_trials": N_TRIALS,
    "mlflow_experiment": EXPERIMENT_NAME,
    "study_name": STUDY_PATH.stem,
    "decision_rule": "classifier.predict",
    "cpu_simd_found": CPU_SIMD_FOUND,
}

Add the searched ranges, the fixed hyperparameters and, in a smoke run, the number of rows read from each file

In [ ]:
for name, space in VECTORIZER_SEARCH_SPACE.items():
    run_settings[f"search.{name}"] = space
for name, space in CLASSIFIER_SEARCH_SPACE.items():
    run_settings[f"search.{name}"] = space
for name, value in classifiers.CLASSIFIER_FIXED_PARAMS[CLASSIFIER].items():
    run_settings[f"fixed.{name}"] = value
if SMOKE:
    for split, n_rows in SMOKE_ROWS.items():
        run_settings[f"smoke_rows.{split}"] = n_rows

The study also stores the hashes of the data files and of uv.lock, because the token counts depend on both

In [ ]:
study_settings = dict(run_settings)
study_settings["uv_lock_sha256"] = run_context.environment["uv_lock_sha256"]
for path in CLEANED_PATHS.values():
    hash_key = f"data_sha256.{path.name}"
    study_settings[hash_key] = run_context.environment[hash_key]

A study or a result folder with different settings stops the run here, before anything is counted, trained or written.

In [ ]:
runs.check_study_settings(STUDY_PATH, study_settings)
# find_finished_run refuses a result folder of this run with other settings;
# a finished one without its study would run the whole search again
if runs.find_finished_run(RUN_DIR, run_settings) and not STUDY_PATH.is_file():
    raise RuntimeError(
        f"{RUN_DIR} exists but its study {STUDY_PATH.name} does not - move the folder away.",
    )